In [ ]:
import kagglehub
briscdataset_brisc2025_path = kagglehub.dataset_download('briscdataset/brisc2025')

print('Data source import complete.')

In [ ]:
!pip install -q segmentation-models-pytorch==0.3.4 albumentations==1.4.3
!pip install -q opencv-contrib-python-headless  # for SIFT, Gabor, etc.

In [ ]:
import kagglehub
briscdataset_brisc2025_path = kagglehub.dataset_download('briscdataset/brisc2025')
print('Data source import complete.')

In [ ]:
import os, glob, random, warnings, hashlib
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
DATA_ROOT      = "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025"
SEG_TRAIN_IMG  = os.path.join(DATA_ROOT, "segmentation_task", "train", "images")
SEG_TRAIN_MASK = os.path.join(DATA_ROOT, "segmentation_task", "train", "masks")
SEG_TEST_IMG   = os.path.join(DATA_ROOT, "segmentation_task", "test",  "images")
SEG_TEST_MASK  = os.path.join(DATA_ROOT, "segmentation_task", "test",  "masks")

IMG_SIZE = 224
MEAN     = np.array([0.485, 0.456, 0.406])
STD      = np.array([0.229, 0.224, 0.225])

def build_seg_df(img_dir, mask_dir):
    rows = []
    for img_path in sorted(glob.glob(os.path.join(img_dir, "*.jpg"))):
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(mask_dir, stem + ".png")
        if os.path.exists(mask_path):
            rows.append({"img_path": img_path, "mask_path": mask_path})
    return pd.DataFrame(rows)

seg_train_df = build_seg_df(SEG_TRAIN_IMG, SEG_TRAIN_MASK)
seg_test_df  = build_seg_df(SEG_TEST_IMG,  SEG_TEST_MASK)
print("Seg train:", len(seg_train_df), "| Seg test:", len(seg_test_df))

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("Checking for data leakage between train and test images...")
train_hashes = {file_hash(p): p for p in seg_train_df["img_path"]}
duplicates   = [p for p in seg_test_df["img_path"] if file_hash(p) in train_hashes]
print(f"Duplicates found: {len(duplicates)}")
seg_test_df = seg_test_df[~seg_test_df["img_path"].isin(duplicates)].reset_index(drop=True)
print(f"Test after cleaning: {len(seg_test_df)}")

In [ ]:
# Each feature function takes a uint8 grayscale image (H, W)
# and returns a float32 array in [0, 1] of shape (H, W).

def feat_clahe(gray):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    out   = clahe.apply(gray)
    return out.astype(np.float32) / 255.0

def feat_sobel(gray):
    sx  = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy  = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    mag = np.clip(mag / mag.max() if mag.max() > 0 else mag, 0, 1)
    return mag.astype(np.float32)

def feat_laplacian(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap = np.abs(lap)
    lap = np.clip(lap / lap.max() if lap.max() > 0 else lap, 0, 1)
    return lap.astype(np.float32)

def feat_gabor(gray, ksize=21, sigma=4, theta=np.pi/4, lambd=10, gamma=0.5):
    kernel = cv2.getGaborKernel((ksize, ksize), sigma, theta, lambd, gamma, 0, ktype=cv2.CV_32F)
    out    = cv2.filter2D(gray.astype(np.float32), cv2.CV_32F, kernel)
    out    = np.abs(out)
    out    = out / out.max() if out.max() > 0 else out
    return out.astype(np.float32)

def feat_harris(gray):
    gray_f = np.float32(gray)
    dst    = cv2.cornerHarris(gray_f, blockSize=2, ksize=3, k=0.04)
    dst    = np.clip(dst / dst.max() if dst.max() > 0 else dst, 0, 1)
    return dst.astype(np.float32)

def feat_canny(gray):
    edges = cv2.Canny(gray, 50, 150)
    return edges.astype(np.float32) / 255.0

def feat_sift_density(gray):
    """SIFT keypoint response as a dense heatmap (Gaussian-blurred)."""
    sift    = cv2.SIFT_create()
    kps     = sift.detect(gray, None)
    heatmap = np.zeros(gray.shape, dtype=np.float32)
    for kp in kps:
        x, y = int(kp.pt[0]), int(kp.pt[1])
        if 0 <= y < gray.shape[0] and 0 <= x < gray.shape[1]:
            heatmap[y, x] += kp.response
    heatmap = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=5)
    heatmap = heatmap / heatmap.max() if heatmap.max() > 0 else heatmap
    return heatmap

def feat_dog(gray, k=1.6, sigma=1.0):
    """Difference of Gaussians — dense approximation of SIFT scale-space."""
    g1  = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), sigma)
    g2  = cv2.GaussianBlur(gray.astype(np.float32), (0, 0), sigma * k)
    dog = np.abs(g1 - g2)
    dog = dog / dog.max() if dog.max() > 0 else dog
    return dog.astype(np.float32)

def feat_lbp(gray, radius=1, n_points=8):
    """Local Binary Pattern — texture descriptor."""
    from skimage.feature import local_binary_pattern
    lbp = local_binary_pattern(gray, n_points, radius, method='uniform')
    lbp = lbp / lbp.max() if lbp.max() > 0 else lbp
    return lbp.astype(np.float32)

# Registry: name → function
FEATURE_REGISTRY = {
    "clahe":        feat_clahe,
    "sobel":        feat_sobel,
    "laplacian":    feat_laplacian,
    "gabor":        feat_gabor,
    "harris":       feat_harris,
    "canny":        feat_canny,
    "sift_density": feat_sift_density,
    "dog":          feat_dog,
    "lbp":          feat_lbp,
}

print("Available features:", list(FEATURE_REGISTRY.keys()))

In [ ]:
# ============================================================
# CELL 5 — ⚙️ CONFIGURATION CELL (edit this each run)
# ============================================================
# Pick any 2 keys from FEATURE_REGISTRY above.
# Channel layout: [grayscale | FEATURE_A | FEATURE_B]

FEATURE_A = "sift_density"        # e.g. "clahe", "sobel", "laplacian", "gabor",
FEATURE_B = "dog"        #      "harris", "canny", "sift_density", "dog", "lbp"

SEG_MODEL_NAME    = "UNet-EffB0"   # see SEG_REGISTRY below

# Recommended settings for your ~6k dataset (single phase, lower overfitting risk)
LR_ENCODER        = 5e-5
LR_DECODER        = 1e-4
WEIGHT_DECAY      = 1e-3
TOTAL_EPOCHS      = 25
EARLY_STOP_PATIENCE = 6

SEG_REGISTRY = {
    "UNet-EffB0":          (smp.Unet,          "efficientnet-b0"),
    "UNet-ResNet34":       (smp.Unet,          "resnet34"),
    "UNetPP-EffB0":        (smp.UnetPlusPlus,  "efficientnet-b0"),
    "UNetPP-ResNet34":     (smp.UnetPlusPlus,  "resnet34"),
    "DeepLabV3P-EffB0":    (smp.DeepLabV3Plus, "efficientnet-b0"),
    "DeepLabV3P-ResNet50": (smp.DeepLabV3Plus, "resnet50"),
    "FPN-EffB0":           (smp.FPN,           "efficientnet-b0"),
    "MAnet-EffB0":         (smp.MAnet,         "efficientnet-b0"),
}

print(f"Config: [{FEATURE_A}] + [{FEATURE_B}] | Model: {SEG_MODEL_NAME}")
print(f"Encoder LR: {LR_ENCODER} | Decoder LR: {LR_DECODER} | Weight Decay: {WEIGHT_DECAY}")
print(f"Total epochs: {TOTAL_EPOCHS} | Early stopping patience: {EARLY_STOP_PATIENCE}")

In [ ]:
# ============================================================
# CELL 6 — Feature Cache & Generation
# ============================================================
# Cache is a dict keyed by (img_path, feature_name).
# Only computed once per session per (path, feature) pair.

_FEATURE_CACHE = {}

def get_feature(img_path: str, feature_name: str, target_size: int) -> np.ndarray:
    """Return a (H, W) float32 feature map, computing + caching if needed."""
    key = (img_path, feature_name, target_size)
    if key in _FEATURE_CACHE:
        return _FEATURE_CACHE[key]
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    gray = cv2.resize(gray, (target_size, target_size))
    feat = FEATURE_REGISTRY[feature_name](gray)
    _FEATURE_CACHE[key] = feat
    return feat

def build_3ch_image(img_path: str, feat_a: str, feat_b: str, target_size: int) -> np.ndarray:
    """
    Returns (H, W, 3) float32 array:
      ch0 = grayscale original (normalized to [0,1])
      ch1 = feature A
      ch2 = feature B
    """
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    gray = cv2.resize(gray, (target_size, target_size))
    ch0  = gray.astype(np.float32) / 255.0
    ch1  = get_feature(img_path, feat_a, target_size)
    ch2  = get_feature(img_path, feat_b, target_size)
    return np.stack([ch0, ch1, ch2], axis=-1)  # (H, W, 3)

print("Feature cache and builder ready.")

In [ ]:
# ============================================================
# CELL 7 — Augmentations & Datasets
# ============================================================
# No data augmentation (only Normalize + ToTensor)

seg_aug_train = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

seg_aug_test = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class BrainSegDataset(Dataset):
    def __init__(self, df, feat_a, feat_b, transform=None):
        self.df        = df.reset_index(drop=True)
        self.feat_a    = feat_a
        self.feat_b    = feat_b
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = build_3ch_image(row["img_path"], self.feat_a, self.feat_b, IMG_SIZE)
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)

        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = (mask > 127).astype(np.uint8)

        if self.transform:
            aug       = self.transform(image=img_u8, mask=mask)
            img_u8, mask = aug["image"], aug["mask"]

        return img_u8, mask.long()

seg_train_ds = BrainSegDataset(seg_train_df, FEATURE_A, FEATURE_B, seg_aug_train)
seg_test_ds  = BrainSegDataset(seg_test_df,  FEATURE_A, FEATURE_B, seg_aug_test)

seg_train_dl = DataLoader(seg_train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
seg_test_dl  = DataLoader(seg_test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(seg_train_dl)} | Test batches: {len(seg_test_dl)}")
print("Data augmentation: DISABLED (only normalization)")

In [ ]:
# ============================================================
# CELL 9 — Loss & Metrics
# ============================================================
bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary")

def seg_criterion(pred, target):
    target_f = target.float().unsqueeze(1)
    return 0.5 * bce_loss(pred, target_f) + 0.5 * dice_loss(pred, target_f)

def compute_metrics(pred_logits, target):
    """
    Returns dict with: dice, iou
    pred_logits: (B, 1, H, W)  raw logits
    target:      (B, H, W)     long mask
    """
    pred  = (torch.sigmoid(pred_logits) > 0.5).float()
    tgt   = target.float().unsqueeze(1)

    inter = (pred * tgt).sum(dim=(2, 3))
    union_dice = pred.sum(dim=(2, 3)) + tgt.sum(dim=(2, 3))
    union_iou  = pred.sum(dim=(2, 3)) + tgt.sum(dim=(2, 3)) - inter

    dice = ((2 * inter + 1e-6) / (union_dice + 1e-6)).mean().item()
    iou  = ((inter + 1e-6) / (union_iou + 1e-6)).mean().item()

    return {"dice": dice, "iou": iou}

print("Loss and metrics defined.")

In [ ]:
# ============================================================
# CELL 10 — Train / Eval Loop Helpers
# ============================================================
def train_one_epoch_seg(model, loader, optimizer):
    model.train()
    total_loss = total_dice = total_iou = n = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = seg_criterion(pred, masks)
        loss.backward()
        optimizer.step()
        m             = compute_metrics(pred, masks)
        total_loss   += loss.item() * imgs.size(0)
        total_dice   += m["dice"]  * imgs.size(0)
        total_iou    += m["iou"]   * imgs.size(0)
        n            += imgs.size(0)
    return total_loss / n, total_dice / n, total_iou / n

@torch.no_grad()
def eval_seg(model, loader):
    model.eval()
    total_loss = total_dice = total_iou = n = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pred         = model(imgs)
        m            = compute_metrics(pred, masks)
        total_loss  += seg_criterion(pred, masks).item() * imgs.size(0)
        total_dice  += m["dice"] * imgs.size(0)
        total_iou   += m["iou"]  * imgs.size(0)
        n           += imgs.size(0)
    return total_loss / n, total_dice / n, total_iou / n

print("Train/eval helpers ready.")

In [ ]:
# ============================================================
# CELL 8 — Model Helpers
# ============================================================
def unfreeze_seg_encoder(model):
    """Make all parameters trainable"""
    for param in model.parameters():
        param.requires_grad = True
    print(f"  All parameters are now trainable.")

print("Model helper defined.")

In [ ]:
# ============================================================
# CELL 11 — Build Model
# ============================================================
arch, encoder = SEG_REGISTRY[SEG_MODEL_NAME]
seg_model = arch(
    encoder_name    = encoder,
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 1,
    activation      = None,
).to(DEVICE)

unfreeze_seg_encoder(seg_model)

# Discriminative learning rates (recommended for your case)
seg_optimizer = torch.optim.AdamW([
    {"params": seg_model.encoder.parameters(), "lr": LR_ENCODER},
    {"params": seg_model.decoder.parameters(), "lr": LR_DECODER},
], weight_decay=WEIGHT_DECAY)

print(f"Model built: {SEG_MODEL_NAME} | Encoder: {encoder}")
print("Using discriminative learning rates (encoder slow, decoder normal)")

In [ ]:
# ============================================================
# CELL 12 — Single Phase Training with Early Stopping
# ============================================================
history = {
    "train_loss": [], "val_loss": [],
    "train_dice": [], "val_dice": [],
    "train_iou":  [], "val_iou":  [],
}

best_val_dice = 0.0
patience_counter = 0

print(f"\n── Starting Training ──")
print(f"Model: {SEG_MODEL_NAME} | Features: gray + {FEATURE_A} + {FEATURE_B}")
print(f"Total epochs: {TOTAL_EPOCHS} | Early stop patience: {EARLY_STOP_PATIENCE}\n")

for epoch in range(1, TOTAL_EPOCHS + 1):
    tr_loss, tr_dice, tr_iou = train_one_epoch_seg(seg_model, seg_train_dl, seg_optimizer)
    vl_loss, vl_dice, vl_iou = eval_seg(seg_model, seg_test_dl)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_dice"].append(tr_dice)
    history["val_dice"].append(vl_dice)
    history["train_iou"].append(tr_iou)
    history["val_iou"].append(vl_iou)

    flag = ""
    if vl_dice > best_val_dice:
        best_val_dice = vl_dice
        torch.save(seg_model.state_dict(), f"best_seg_{SEG_MODEL_NAME}.pth")
        flag = " ✓"
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"[{epoch:02d}/{TOTAL_EPOCHS}] "
          f"loss={tr_loss:.4f}/{vl_loss:.4f}  "
          f"dice={tr_dice:.4f}/{vl_dice:.4f}  "
          f"iou={tr_iou:.4f}/{vl_iou:.4f}{flag}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest validation Dice: {best_val_dice:.4f}")

In [ ]:
# ============================================================
# CELL 14 — Training Curves
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs, history["train_dice"], label="train")
axes[1].plot(epochs, history["val_dice"], label="val")
axes[1].set_title("Dice Coefficient")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs, history["train_iou"], label="train")
axes[2].plot(epochs, history["val_iou"], label="val")
axes[2].set_title("IoU")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.suptitle(f"{SEG_MODEL_NAME} | gray + {FEATURE_A} + {FEATURE_B}", fontsize=14)
plt.tight_layout()
plt.savefig(f"seg_curves_{SEG_MODEL_NAME}.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 15 — Final Test Evaluation & Summary
# ============================================================
seg_model.load_state_dict(torch.load(f"best_seg_{SEG_MODEL_NAME}.pth"))
_, final_dice, final_iou = eval_seg(seg_model, seg_test_dl)

print("=" * 60)
print(f"FINAL RESULTS")
print(f"  Model          : {SEG_MODEL_NAME}")
print(f"  Features       : gray + {FEATURE_A} + {FEATURE_B}")
print(f"  Test Dice      : {final_dice:.4f}")
print(f"  Test IoU       : {final_iou:.4f}")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 16 — Visualization: Regular + Fancy Overlay
# ============================================================
import matplotlib.patches as mpatches

seg_model.load_state_dict(torch.load(f"best_seg_{SEG_MODEL_NAME}.pth"))
seg_model.eval()

num_samples = 4
indices = random.sample(range(len(seg_test_ds)), num_samples)

fig, axes = plt.subplots(num_samples, 5, figsize=(20, 5 * num_samples))
fig.suptitle(f"Visualization — {SEG_MODEL_NAME} | gray + {FEATURE_A} + {FEATURE_B}", fontsize=16, y=0.95)

for i, idx in enumerate(indices):
    img_tensor, mask_gt = seg_test_ds[idx]
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_logits = seg_model(img_tensor)
        pred_prob   = torch.sigmoid(pred_logits[0, 0]).cpu().numpy()
        pred_mask   = (pred_prob > 0.5).astype(np.uint8)

    orig_img = seg_test_ds.df.iloc[idx]["img_path"]
    orig     = cv2.imread(orig_img, cv2.IMREAD_GRAYSCALE)
    orig     = cv2.resize(orig, (IMG_SIZE, IMG_SIZE))

    mask_gt_np = mask_gt.cpu().numpy()

    # Col 0: Original
    axes[i, 0].imshow(orig, cmap='gray')
    axes[i, 0].set_title("Original Image")
    axes[i, 0].axis('off')

    # Col 1: Ground Truth
    axes[i, 1].imshow(mask_gt_np, cmap='gray')
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis('off')

    # Col 2: Predicted Mask
    axes[i, 2].imshow(pred_mask, cmap='gray')
    axes[i, 2].set_title("Predicted Mask")
    axes[i, 2].axis('off')

    # Col 3: Simple Overlay (red prediction on grayscale)
    overlay_simple = np.stack([orig, orig, orig], axis=-1).astype(np.float32) / 255.0
    overlay_simple[..., 0] = np.maximum(overlay_simple[..., 0], pred_mask * 0.8)
    axes[i, 3].imshow(overlay_simple)
    axes[i, 3].set_title("Simple Overlay (Red)")
    axes[i, 3].axis('off')

    # Col 4: Fancy Overlay (green, semi-transparent)
    overlay_fancy = np.stack([orig, orig, orig], axis=-1).astype(np.float32) / 255.0
    green_layer   = np.stack([
        np.zeros_like(pred_mask, dtype=np.float32),
        pred_mask.astype(np.float32) * 0.9,
        np.zeros_like(pred_mask, dtype=np.float32),
    ], axis=-1)
    overlay_fancy = cv2.addWeighted(overlay_fancy, 0.7, green_layer, 0.5, 0)

    axes[i, 4].imshow(overlay_fancy)
    axes[i, 4].set_title("Fancy Overlay")
    axes[i, 4].axis('off')

    green_patch = mpatches.Patch(color='lime', label='Predicted Region')
    axes[i, 4].legend(handles=[green_patch], loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(f"visualization_{SEG_MODEL_NAME}.png", dpi=200, bbox_inches='tight')
plt.show()